In [1]:
from pathlib import Path
from typing import Callable

import prism
import xarray as xr

from imagematerials.stock import compute_dynamic_stock_driven, compute_historic
from imagematerials.survival import ScipySurvival, SurvivalMatrix
from imagematerials.vehicles.preprocessing import (
    export_to_netcdf,
    import_from_netcdf,
    preprocessing,
)


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [33]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)

In [4]:
vehicle_nr = prep_data['total_nr_vehicles']
life_time_vehicles = prep_data["lifetimes_vehicles"]
vehicle_shares = prep_data["vehicle_shares"]

In [5]:
survival_matrix = SurvivalMatrix(ScipySurvival(life_time_vehicles, vehicle_nr.coords["mode"]))

In [6]:
start_simulation = 1970
end_simulation = vehicle_nr.coords["time"].max()
Region = prism.Dimension("region", coords=[str(x) for x in vehicle_nr.coords["region"].values])
Mode = prism.Dimension("mode", coords=[str(x) for x in vehicle_nr["mode"].to_numpy()])
Cohort = prism.Dimension("cohort", coords=vehicle_shares.coord["cohort"])
stock_by_cohort = xr.DataArray(0.0, dims=("time", "cohort", "region", "mode"),
                                coords={"time": vehicle_nr.coords["time"],
                                        "cohort": vehicle_nr.coords["cohort"],
                                        "region": vehicle_nr.coords["region"],
                                        "mode": vehicle_nr.coords["mode"]})


In [7]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    stock: prism.TimeVariable[Region, Mode] #TODO check how to have property that can be both input and output within prism
    stock_function: Callable    # defines the stock function to use e.g. stock or inflow driven
    stock_by_cohort: xr.DataArray
    shares: prism.TimeVariable[Region, Mode]

    # stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.stock, self.survival_matrix, self.start_simulation,
                        self.stock_by_cohort, self.inflow, self.outflow_by_cohort,
                        self.shares,
                        self.stock_function)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        self.stock_function(self.stock, self.stock_by_cohort,  self.inflow, self.outflow_by_cohort,
                            self.survival_matrix, t, self.shares)


In [8]:
timeline = prism.Timeline(start=vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [9]:
model = Stocks(timeline, start_simulation=start_simulation, survival_matrix=survival_matrix,
                stock=vehicle_nr, stock_function = compute_dynamic_stock_driven,
                stock_by_cohort=stock_by_cohort, shares=vehicle_shares)

In [10]:
model.simulate(timeline_simulate)

In [19]:
from pprint import pprint
pprint(dict(zip(model.stock_by_cohort.loc[2020].sum("region").sum("cohort").values, model.stock_by_cohort.coords["mode"].values)))

{np.float64(0.0): np.str_('Regular Buses - PHEV'),
 np.float64(1.3128623249764254e-14): np.str_('Heavy Freight Trucks - FCV'),
 np.float64(1.4725771878123492e-10): np.str_('Medium Freight Trucks - FCV'),
 np.float64(8.876797729756504e-10): np.str_('Light Commercial Vehicles - FCV'),
 np.float64(9.676695269055972e-05): np.str_('Midi Buses - BEV'),
 np.float64(0.0005054144556248017): np.str_('Regular Buses - BEV'),
 np.float64(393.3913118069255): np.str_('Cars - FCV'),
 np.float64(4120.646347478642): np.str_('Very Large Ships'),
 np.float64(9908.267224614654): np.str_('High Speed Trains'),
 np.float64(9958.917226434349): np.str_('Large Ships'),
 np.float64(31080.28191340368): np.str_('Small Ships'),
 np.float64(37927.606363664505): np.str_('Passenger Planes'),
 np.float64(38401.27014341562): np.str_('Medium Ships'),
 np.float64(62404.96687299153): np.str_('Midi Buses - HEV'),
 np.float64(82962.85590236567): np.str_('Medium Freight Trucks - HEV'),
 np.float64(98419.02965883589): np.str_('

In [26]:
dict(model.inflow[2020].coords.items())

{'time': <xarray.DataArray 'time' ()> Size: 8B
array(2020)
Coordinates:
    time     int64 8B 2020, 'mode': <xarray.DataArray 'mode' (mode: 53)> Size: 7kB
array(['Bikes', 'Cars', 'Cars - BEV', 'Cars - FCV', 'Cars - HEV', 'Cars - ICE',
       'Cars - PHEV', 'Cars - Trolley', 'Freight Planes', 'Freight Trains',
       'Heavy Freight Trucks', 'Heavy Freight Trucks - BEV',
       'Heavy Freight Trucks - FCV', 'Heavy Freight Trucks - HEV',
       'Heavy Freight Trucks - ICE', 'Heavy Freight Trucks - PHEV',
       'Heavy Freight Trucks - Trolley', 'High Speed Trains', 'Inland Ships',
       'Large Ships', 'Light Commercial Vehicles',
       'Light Commercial Vehicles - BEV', 'Light Commercial Vehicles - FCV',
       'Light Commercial Vehicles - HEV', 'Light Commercial Vehicles - ICE',
       'Light Commercial Vehicles - PHEV',
       'Light Commercial Vehicles - Trolley', 'Medium Freight Trucks',
       'Medium Freight Trucks - BEV', 'Medium Freight Trucks - FCV',
       'Medium Freight Truc

In [31]:
def _convert_timevar(time_var, time_coor):
    random_t = time_coor.coords["time"].values[0]
    coords = dict(time_var[random_t].coords.items())
    coords["time"] = time_coor
    array = xr.DataArray(0.0, dims=list(coords), coords=coords)
    for t in time_coor.values:
        array.loc[{"time": t}] = time_var[t]
    return array

In [32]:
_convert_timevar(model.inflow, vehicle_nr.coords["time"])

/home/qubix/.pyenv/versions/3.10.13/lib/python3.10/site-packages/xarray/core/indexing.py:1522: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


<xarray.DataArray (region: 28, mode: 53, time: 254)> Size: 3MB
array([[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         3.94071783e+05, 2.57549798e+05, 2.47026840e+05],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         2.25426682e+06, 2.26961826e+06, 2.27067812e+06],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.26174763e+06, 1.27994169e+06, 1.28731555e+06],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         3.97141229e-01, 5.57562383e+00, 4.68429313e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         8.20522591e+00, 8.47127162e+00, 1.69951410e+01],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.29740152e+00, 1.37810031e+00, 1.23978321e+00]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.89352601e+06, 2.04909772e+06, 1.79285612e+06],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         5.45229824e+05, 5.61099934e+05, 5.76563967e+05],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.88953049e+05, 1.97525300e+05, 2.05640995e+05],
...
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.77277282e+01, 1.75629613e+01, 1.47062275e+01],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         5.10110875e+01, 5.41826276e+01, 5.73529040e+01],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.26723970e+01, 1.24999406e+01, 1.22764303e+01]],

       [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         2.54456379e+07, 1.55745234e+07, 4.91053168e+06],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         7.55379672e+06, 7.76577030e+06, 7.97935486e+06],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         9.03581387e+05, 1.01837671e+06, 1.14180818e+06],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         3.99501900e+00, 4.10981415e+00, 3.45175950e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.90251245e+02, 1.82187320e+02, 2.04347843e+02],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         6.22816260e+00, 6.17578342e+00, 6.13280974e+00]]])
Coordinates:
  * region   (region) <U2 224B '1' '10' '11' '12' '13' ... '5' '6' '7' '8' '9'
  * mode     (mode) <U35 7kB 'Bikes' 'Cars' ... 'Trains' 'Very Large Ships'
  * time     (time) int64 2kB 1807 1808 1809 1810 1811 ... 2057 2058 2059 2060